# Phase 4: when can an interval estimate be used?

This notebook is a readable companion to the deterministic Python pipeline, not a second implementation of training. Run it from the repository root after `python -m src.flight_fuel.phase4`. It reads the committed aggregate report; raw trajectories, labels, and trained models remain in ignored local data. The full commands and frozen decisions are in `docs/phase-4-results.md` and `configs/phase-4-protocol.json`.

**Question.** For completed ACARS-defined intervals, can a rule based on permitted metadata and ADS-B observations identify estimates with lower absolute error at a stated acceptance rate? Abstaining leaves an interval without an estimate. All comparisons here measure agreement with the released ACARS-derived labels, not true engine-flow accuracy.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

report = json.loads(Path("reports/phase-4-results.json").read_text())
report["block_counts"]


## Inputs and separation

April–August trains the fixed ADS-B HistGB model and aircraft-type median-rate comparator. September 1–9 trains a second HistGB model to predict the absolute residual of the zero-ADS-B fallback. September 10–18 sets acceptance thresholds using input scores only. September 19–27 is a later validation block; three flights touching boundaries are excluded. October is evaluated once with these rules frozen. A whole flight belongs to one block. The risk model uses type, duration, ADS-B counts, the largest observation gap, and the base predicted fuel. It never receives the interval fuel label when making an estimate. Integer-pound label-lattice membership is diagnostic only.

The default estimate switches from ADS-B HistGB to the type-rate baseline when the interval has no usable ADS-B positions. The selected estimate is reported only when the risk score is below the calibrated cutoff. A 75% target acceptance is the primary operating point. Scores are in kilograms per interval.


In [ ]:
rows = []
for period in ("validation", "final"):
    p = report["periods"][period]
    for name, metrics in p["models"].items():
        rows.append({"period": period, "model": name,
                     "MAE kg": metrics["mae_kg"], "RMSE kg": metrics["rmse_kg"]})
pd.DataFrame(rows).round(2)


## Risk and coverage

The random, duration and maximum-gap rules are comparators with thresholds calibrated on the same September block. Equal target coverage need not produce identical realized coverage later. Lower accepted-case MAE alone does not mean a model improves every interval; it may choose easier flights. Examine coverage by aircraft and interval duration too.


In [ ]:
rows = []
for period in ("validation", "final"):
    p = report["periods"][period]
    for rule, targets in p["selectors"].items():
        for target, value in targets.items():
            rows.append({"period": period, "rule": rule, "target": float(target),
                         "actual coverage": value["accepted_fraction"],
                         "accepted MAE kg": value["accepted_score"]["mae_kg"]})
coverage = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, period in zip(axes, ("validation", "final")):
    for rule, group in coverage[coverage.period.eq(period)].groupby("rule"):
        ax.plot(group["actual coverage"], group["accepted MAE kg"], marker="o", label=rule)
    ax.set(title=period.title(), xlabel="Fraction of intervals accepted", xlim=(0.4, 1.02))
axes[0].set_ylabel("MAE on accepted intervals (kg)")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
primary = report["periods"]["final"]["selectors"]["learned_risk"]["0.75"]
for group in ("aircraft_type", "duration_bin", "adsb_input"):
    print("\n", group)
    print(pd.DataFrame(primary["acceptance_by_group"][group]).T.round(3))


## Controlled ADS-B outages

Within a fixed seeded sample of September validation flights, raw ADS-B rows are removed **before** recomputing interval features. Randomly scattered rows and one contiguous point block are each removed at three fractions and two seeds. The training data, label, metadata, and aircraft type stay fixed; the scored set contains only intervals that originally had at least one ADS-B position. A contiguous *point* block is not a fixed-duration outage. Bootstrap intervals resample whole flights and quantify uncertainty in the paired change from the same sample’s natural input.


In [ ]:
outage = report["outage_experiment"]
rows = [{"scenario": name, "ADS-B MAE kg": value["adsb"]["mae_kg"],
         "fallback MAE kg": value["fallback"]["mae_kg"],
         "zero ADS-B share": value["zero_adsb_fraction"],
         "change vs natural kg": value["adsb_vs_natural"]["second_minus_first_mae_kg"]}
        for name, value in outage["scenarios"].items()]
print("Sample:", outage["flights"], "flights,", outage["intervals"], "originally observed intervals")
print("Natural ADS-B MAE:", round(outage["natural_adsb"]["mae_kg"], 2), "kg")
pd.DataFrame(rows).round(2)


## Label granularity diagnostic

The lattice flag detects values close to an integer number of pounds after division by 0.453592. It does not establish how a label was computed and is never used to fit or select a model. Different error between the two groups can reflect aircraft, interval length, scale, or label processing; it is not a measurement-noise estimate.


In [ ]:
pd.DataFrame({period: report["periods"][period]["label_granularity"]["all"]["all"]
              for period in ("validation", "final")}).T.round(3)


## Interpretation

The October 75% risk selector accepts almost no 15–60 minute intervals and very few B744 intervals. Its low accepted-case MAE is therefore unsuitable as a universal accuracy claim. Compare the aircraft and duration tables before proposing any use. A deployment decision also requires a defined cost for abstention, prospective access to interval boundaries, and validation on independent operators and observation systems. The published data do not provide those tests. The linked report records numerical results and the exact reproduction commands.
